In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import numpy as np

In [3]:
df = pd.read_csv('insurance.csv')

In [4]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
49,57,76.6,1.61,11.7,yes,Chennai,Doctor,high
40,35,55.9,1.71,28.3,no,Jaipur,Business,low
54,42,71.8,1.71,11.5,yes,Pune,Business,medium
59,43,83.1,1.74,17.5,yes,Kolkata,Driver,high
55,32,60.2,1.63,21.8,no,Bangalore,Farmer,low


In [5]:
df_feat = df.copy()

In [6]:
#Feature 1: BMI
df_feat["bmi"] = df_feat["weight"]/(df_feat["height"]**2)

In [7]:
#Feature 2: Age Group
def age_group(age):
  if age < 25:
    return "young"
  elif age < 45:
    return "adult"
  elif age < 60:
    return "middle_aged"
  return "senior"

In [8]:
df_feat["age_group"] = df_feat["age"].apply(age_group)

In [9]:
#Feature 3: Lifestyle Risk
def lifestyle_risk(row):
  if row["smoker"] and row["bmi"] > 30:
    return "high"
  elif row["smoker"] or row["bmi"] > 27:
    return "medium"
  else:
    return "low"

In [10]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [11]:
tier_1_cities = ["Mumbai","Delhi","Bangalore","Chennai","Hyderabad", "Pune"]

tier_2_cities = ["Jaipur", "chandigarh","Indore","Lucknow","Patna","Ranchi","Visakhapatnam","Coimbatore","Bhopal","Nagpur","Vadodara","Surat","Rajkot","Jodhpur","Raipur","Amritsar","Varanasi","Agra","Dehradun","Mysore","Jabalpur","Guwahati","Thiruvananthapuram", "Ludhiana","Nashik","Prayagraj","Udaipur","SambhajiNagar","Hubli","Belgaum","Salem","Vijayawada","Tiruchirapalli","Bhavnagar","Gwalior","Dhanbad","Bareilly","Aligarh","Gaya","kozhikode","Warangal","Kolhapur","Bilaspur","Jalandhar","Noida","Guntur","Asansol","Siliguri"]

In [12]:
# Feature 4: City Tier
def city_tier(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2
  else:
    return 3

In [13]:
df_feat["city_tier"] = df_feat["city"].apply(city_tier)

In [14]:
df_feat.drop(columns=["age", "weight", "height", "smoker", "city"])[['income_lpa',"occupation","bmi","age_group","lifestyle_risk","city_tier","insurance_premium_category"]]

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
0,31.6,Engineer,26.056031,middle_aged,medium,1,medium
1,14.7,Business,20.272399,adult,medium,1,medium
2,8.9,Doctor,26.164901,young,medium,1,medium
3,17.2,Farmer,22.283737,adult,medium,1,medium
4,25.5,Driver,23.348923,adult,medium,1,low
...,...,...,...,...,...,...,...
95,24.1,Teacher,26.609713,middle_aged,medium,1,medium
96,39.5,Engineer,22.127899,adult,medium,1,medium
97,33.5,Farmer,29.752066,adult,medium,2,medium
98,12.4,Driver,20.400000,middle_aged,medium,1,medium


In [15]:
# select features and targets
X = df_feat[["bmi","age_group","lifestyle_risk","city_tier","income_lpa","occupation"]]
y = df_feat["insurance_premium_category"]

In [16]:
X

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,26.056031,middle_aged,medium,1,31.6,Engineer
1,20.272399,adult,medium,1,14.7,Business
2,26.164901,young,medium,1,8.9,Doctor
3,22.283737,adult,medium,1,17.2,Farmer
4,23.348923,adult,medium,1,25.5,Driver
...,...,...,...,...,...,...
95,26.609713,middle_aged,medium,1,24.1,Teacher
96,22.127899,adult,medium,1,39.5,Engineer
97,29.752066,adult,medium,2,33.5,Farmer
98,20.400000,middle_aged,medium,1,12.4,Driver


In [17]:
y

,insurance_premium_category
0,medium
1,medium
2,medium
3,medium
4,low
...,...
95,medium
96,medium
97,medium
98,medium


In [18]:
# define categorical and numeric features
categorical_features =["age_group", "lifestyle_risk","occupation","city_tier"]
numeric_features = ["bmi","income_lpa"]

In [25]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [27]:
#create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [28]:
#split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [30]:
#Predit and evaluate
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"Accuracy: {accuracy:.2f}")
print("Classification Report:\n", report)

Accuracy: 0.50
Classification Report:
               precision    recall  f1-score   support

        high       0.40      0.33      0.36         6
         low       0.75      0.50      0.60         6
      medium       0.45      0.62      0.53         8

    accuracy                           0.50        20
   macro avg       0.53      0.49      0.50        20
weighted avg       0.53      0.50      0.50        20



In [31]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
44,28.692653,middle_aged,medium,1,34.8,Teacher
81,24.256055,senior,medium,1,18.2,Business
84,23.163296,young,medium,1,31.5,Farmer
92,20.767316,adult,medium,1,12.5,Freelancer
93,21.755469,adult,medium,2,12.1,Freelancer


In [34]:
import pickle

#Save the trained pipeline using pikle
pickle_model_path = "model.pkl"
with open(pickle_model_path, 'wb') as f:
    pickle.dump(pipeline, f)